# TP Bioinformatique COVID-19 - Partie 1
## Introduction au séquençage haut-débit et formats de données

**Auteur : Marwa zidi**

**Durée estimée : 20 minutes**

### Objectifs
- Comprendre les principes du séquençage Illumina
- Découvrir les formats FASTA et FASTQ
- Interpréter les scores de qualité Phred

---

## 1. Rappels sur le séquençage haut-débit

### Principe du séquençage Illumina

Le séquençage Illumina (NGS - Next Generation Sequencing) repose sur :
1. **Fragmentation** de l'ADN viral
2. **Amplification** par PCR (clusters)
3. **Séquençage par synthèse** avec nucléotides fluorescents
4. **Détection** de fluorescence à chaque cycle

**Caractéristiques :**
- Longueur des reads : 75-300 pb (paires de bases)
- Profondeur : millions de reads
- Précision : ~99.9% (Q30)

![Séquençage Illumina](https://www.illumina.com/content/dam/illumina-marketing/images/science/sequencing-method-explorer/sequencing-by-synthesis-thumbnail.jpg)


In [ ]:
# Import des bibliothèques nécessaires
import os
from Bio import SeqIO
from Bio.Seq import Seq
import matplotlib.pyplot as plt
import numpy as np

print("✅ Bibliothèques importées avec succès")
print(f"📂 Répertoire de travail : {os.getcwd()}")

## 2. Format FASTA

Le format FASTA est le format le plus simple pour stocker des séquences biologiques.

**Structure :**
```
>Identifiant Description
ATCGATCGATCG...
```

### Exemple : Génome de référence SARS-CoV-2

In [ ]:
# Lecture du génome de référence SARS-CoV-2
genome_file = "/opt/covid_data/NC_045512.2.fasta"

if os.path.exists(genome_file):
    genome = SeqIO.read(genome_file, "fasta")
    print(f"🦠 Génome chargé : {genome.id}")
    print(f"📏 Longueur du génome : {len(genome.seq):,} pb")
    print(f"\n🔤 Premiers 100 nucléotides :")
    print(genome.seq[:100])
    
    # Calcul de la composition en bases
    A_count = genome.seq.count('A')
    T_count = genome.seq.count('T')
    G_count = genome.seq.count('G')
    C_count = genome.seq.count('C')
    
    print(f"\n📊 Composition du génome :")
    print(f"   A: {A_count:,} ({A_count/len(genome.seq)*100:.2f}%)")
    print(f"   T: {T_count:,} ({T_count/len(genome.seq)*100:.2f}%)")
    print(f"   G: {G_count:,} ({G_count/len(genome.seq)*100:.2f}%)")
    print(f"   C: {C_count:,} ({C_count/len(genome.seq)*100:.2f}%)")
    print(f"   GC%: {(G_count + C_count)/len(genome.seq)*100:.2f}%")
else:
    print("⚠️ Fichier génome non trouvé")

## 3. Format FASTQ

Le format FASTQ contient en plus des séquences, les **scores de qualité** pour chaque base.

**Structure (4 lignes par read) :**
```
@Identifiant
ATCGATCGATCG...
+
IIIIIIIIIIII...
```

### Scores de qualité Phred (Q)

**Formule :** Q = -10 × log₁₀(P)

où P = probabilité d'erreur

| Score Q | Précision | Erreur | Caractère ASCII |
|---------|-----------|--------|------------------|
| Q10     | 90%       | 1/10   | +                |
| Q20     | 99%       | 1/100  | 5                |
| Q30     | 99.9%     | 1/1000 | ?                |
| Q40     | 99.99%    | 1/10000| I                |


In [ ]:
# Lecture d'un fichier FASTQ (échantillon)
fastq_file = "/opt/covid_data/sample_reads.fastq"

if os.path.exists(fastq_file):
    # Lire les 5 premiers reads
    reads = list(SeqIO.parse(fastq_file, "fastq"))[:5]
    
    print(f"📖 Nombre de reads chargés : {len(reads)}\n")
    
    for i, read in enumerate(reads, 1):
        print(f"🔬 Read #{i}:")
        print(f"   ID: {read.id}")
        print(f"   Séquence: {read.seq[:50]}...")
        print(f"   Qualité: {read.letter_annotations['phred_quality'][:20]}")
        print(f"   Longueur: {len(read.seq)} pb")
        print(f"   Qualité moyenne: {np.mean(read.letter_annotations['phred_quality']):.1f}")
        print()
else:
    print("⚠️ Fichier FASTQ non trouvé")
    print("\n💡 Exemple de read FASTQ :")
    print("""@SRR11140744.1
ATGATGACTGATTACAGATGCAGATTATAGTGCAGTGCATGACTGAT
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII""")

## 4. Exercice pratique

**À faire :**
1. Calculer la longueur moyenne des reads
2. Calculer le score de qualité moyen
3. Identifier le pourcentage de bases avec Q ≥ 30

In [ ]:
# EXERCICE À COMPLÉTER

if os.path.exists(fastq_file):
    reads = list(SeqIO.parse(fastq_file, "fastq"))[:1000]  # Premier millier
    
    # 1. Longueur moyenne
    lengths = [len(read.seq) for read in reads]
    avg_length = np.mean(lengths)
    
    # 2. Score qualité moyen
    all_qualities = []
    for read in reads:
        all_qualities.extend(read.letter_annotations['phred_quality'])
    avg_quality = np.mean(all_qualities)
    
    # 3. Pourcentage Q30
    q30_bases = sum(1 for q in all_qualities if q >= 30)
    percent_q30 = (q30_bases / len(all_qualities)) * 100
    
    print(f"📊 Statistiques sur {len(reads)} reads :")
    print(f"   Longueur moyenne : {avg_length:.1f} pb")
    print(f"   Qualité moyenne : Q{avg_quality:.1f}")
    print(f"   Bases avec Q≥30 : {percent_q30:.2f}%")
    
    # Visualisation de la distribution des qualités
    plt.figure(figsize=(10, 5))
    plt.hist(all_qualities, bins=50, edgecolor='black', alpha=0.7)
    plt.axvline(x=30, color='red', linestyle='--', linewidth=2, label='Q30 threshold')
    plt.xlabel('Score de qualité Phred')
    plt.ylabel('Fréquence')
    plt.title('Distribution des scores de qualité')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("⚠️ Pas de données FASTQ disponibles pour l'exercice")

## 📝 Points clés à retenir

✅ **FASTA** : Format simple (ID + séquence)  
✅ **FASTQ** : Format enrichi (ID + séquence + qualité)  
✅ **Score Q30** : Seuil de qualité standard (99.9% de précision)  
✅ Le génome SARS-CoV-2 fait ~30 kb

---

**➡️ Passez au notebook suivant : `02_Genome_Reference_Annotation.ipynb`**